# AI Coloring Book — Final Prototype

Run the complete name → Wikipedia source → Qwen biography → FLUX line art → PDF pipeline on a free Colab T4. All eight current subjects are loaded from the single `evaluation/subjects.csv` file. Outputs go to a fresh `final_run_v2` directory; old experiments are not modified.


In [ ]:
# Confirm that Colab assigned a GPU. Free T4 and paid L4 runtimes are supported.
!nvidia-smi

In [ ]:
from pathlib import Path

REPOSITORY = 'https://github.com/icynic/AI-coloring-book.git'
PROJECT_DIR = Path('/content/AI-coloring-book')
if not PROJECT_DIR.exists():
    !git clone -q {REPOSITORY} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} pull --ff-only
%cd /content/AI-coloring-book
!git rev-parse --short HEAD

In [ ]:
# Install the final-prototype environment. Pillow is kept within a compatible
# range to avoid replacing Colab's already-loaded PIL modules.
%pip install -q -r requirements-colab.txt

In [ ]:
# Fail early if pip left an incompatible or mixed Colab environment.
from importlib.metadata import version as package_version
from packaging.version import Version

try:
    import PIL
    from PIL import Image, ImageOps
    print('Pillow compatibility check passed:', PIL.__version__)
except ImportError as exc:
    raise RuntimeError(
        'Pillow core modules could not be imported. Re-run the dependency '
        'installation cell, choose Runtime > Restart session, and then run '
        'the notebook again.'
    ) from exc

requests_version = package_version('requests')
protobuf_version = Version(package_version('protobuf'))
if requests_version != '2.32.4' or not Version('5.29.1') <= protobuf_version < Version('6'):
    raise RuntimeError(
        f'Incompatible Colab dependencies: requests={requests_version}, '
        f'protobuf={protobuf_version}. Re-run the install cell using the updated '
        'requirements-colab.txt, then restart the session.'
    )
print('Colab dependency check passed:', requests_version, protobuf_version)

## Configure the run

The current eight-person list replaces Philip I with Otto Hahn and Gertrud von Le Fort with Ferdinand Braun. Generate the complete book from scratch in a new directory. The models load sequentially; the T4 preset uses Qwen 4-bit, FLUX 8-bit and a 640px maximum side. Keep `FORCE_REGENERATE=False` to resume an interrupted run.


In [ ]:
import csv

with Path('evaluation/subjects.csv').open(encoding='utf-8', newline='') as stream:
    NAMES = [row['name'] for row in csv.DictReader(stream)]

USE_GOOGLE_DRIVE = True
T4_SAFE_MODE = True
FORCE_REGENERATE = False
FUZZY_SEARCH = False
SEED = 42
SUMMARY_MIN_WORDS, SUMMARY_MAX_WORDS = 80, 110
QWEN_QUANTIZATION = 'none'   # only used when T4_SAFE_MODE=False
FLUX_QUANTIZATION = 'none'
FLUX_OFFLOAD = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/AIColoringBook/final_run_v2'
else:
    OUTPUT_DIR = '/content/AIColoringBook/final_run_v2'

print('People:', NAMES)
print('Output directory:', OUTPUT_DIR)
print('T4 safe mode:', T4_SAFE_MODE)


In [ ]:
# A fresh unbuffered process displays progress and avoids stale module imports.
import subprocess
import sys

arguments = [
    '--names', *NAMES,
    '--output-dir', OUTPUT_DIR,
    '--seed', str(SEED),
    '--summary-min-words', str(SUMMARY_MIN_WORDS),
    '--summary-max-words', str(SUMMARY_MAX_WORDS),
]
if not FUZZY_SEARCH:
    arguments.append('--no-fuzzy-search')
if T4_SAFE_MODE:
    arguments.append('--t4-safe-mode')
else:
    arguments.extend([
        '--qwen-quantization', QWEN_QUANTIZATION,
        '--flux-quantization', FLUX_QUANTIZATION,
    ])
    if FLUX_OFFLOAD:
        arguments.append('--flux-offload')
if FORCE_REGENERATE:
    arguments.append('--force')

command = [sys.executable, '-u', 'main.py', *arguments]
print('Running:', ' '.join(command), flush=True)

with subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    exit_code = process.wait()

if exit_code != 0:
    raise RuntimeError(
        f'程序退出，错误码 {exit_code}；真正的错误信息见上方输出。'
    )


In [ ]:
# Inspect the reproducibility record and display the finished book link.
import json
from IPython.display import FileLink, display

manifest_path = Path(OUTPUT_DIR) / 'manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps(manifest['runtime'], indent=2))
for item in manifest['items']:
    print(item['title'], 'OK' if not item['errors'] else item['errors'])

if manifest.get('book_path'):
    book_path = Path(OUTPUT_DIR) / Path(manifest['book_path']).name
    display(FileLink(str(book_path)))
else:
    print('No valid book was rebuilt. Inspect the errors above before using old PDFs.')